In [18]:
import re, math
import numpy as np

# --- beállítások ---
EPS = 1e-9
X100_TOL = 0.05     # 5% tűrés a 100x/0.01x felismeréshez
REL_BUCKETS = [1e-6, 1e-2, 1e-1, 1.0]  # határok a rel_err_bucket-hez

# kulcsszókészletek (angol)
PERCENT_WORDS = r"(percent(?:age)?|%|pct)"
RATIO_WORDS   = r"(ratio|proportion|out of|per\s+\w+)"
DIFF_WORDS    = r"(difference|how\s+much(?:\s+(?:bigger|smaller|more|less))?|by\s+how\s+much)"
DIV_WORDS     = r"(divided\s+by|per\s+\d|/)"
ROUND_WORDS   = r"(round(?:ed|ing)?|nearest|approx(?:imate|\.))"

# szám-formátumok
THOUSAND_SEP_RE = re.compile(r"\b\d{1,3}(?:,\d{3})+\b")     # pl. 12,345 vagy 1,234,567
DECIMAL_COMMA_RE = re.compile(r"\b\d+,\d+\b")               # pl. 12,5

def bucketize(value, edges):
    """0..len(edges) közti egész kategória az érték alapján."""
    for i, t in enumerate(edges):
        if value < t:
            return i
    return len(edges)

def magnitude_bucket_of(y_true: float):
    if y_true == 0:
        return -1
    # kerekített nagyságrend, -1: <1, 0: [1,9], 1: [10,99], stb.
    return int(math.floor(math.log10(abs(y_true) + EPS)))

def safe_rel_err(y_true: float, y_pred: float, eps: float = EPS) -> float:
    return abs(y_pred - y_true) / max(abs(y_true), eps)

def approx_ratio(a, b, target, tol):
    """Ellenőrzi, hogy a/b ~ target (relatív tűrés)."""
    if b == 0:
        return False
    r = a / b
    return (1 - tol) * target <= r <= (1 + tol) * target

def extract_flags(question: str, y_true: float, y_pred: float, calc_pattern: str = ""):
    q = question.strip().lower()

    # --- szöveges flag-ek ---
    percent_flag    = int(re.search(PERCENT_WORDS, q) is not None)
    ratio_flag      = int(re.search(RATIO_WORDS, q)   is not None)
    diff_flag       = int(re.search(DIFF_WORDS, q)    is not None)
    div_flag        = int(re.search(DIV_WORDS, q)     is not None)
    rounding_flag   = int(re.search(ROUND_WORDS, q)   is not None)

    has_thousand_sep = int(THOUSAND_SEP_RE.search(question) is not None)
    decimal_comma     = int(DECIMAL_COMMA_RE.search(question) is not None)
    has_percent_sign  = int("%" in question)

    # --- calc_pattern alapú finomítások (ha adsz ilyet, pl. "#/#", "(#-#)/#") ---
    if calc_pattern:
        if "/" in calc_pattern: div_flag = 1
        if "%" in calc_pattern: percent_flag = 1
        if "-" in calc_pattern or "diff" in calc_pattern: diff_flag = 1
        if "ratio" in calc_pattern: ratio_flag = 1

    # --- numerikus jellemzők ---
    mag_bucket   = magnitude_bucket_of(y_true)
    rel_err      = safe_rel_err(y_true, y_pred, EPS)
    rel_err_bucket = bucketize(rel_err, REL_BUCKETS)
    sign_err     = int((y_true * y_pred) < 0)

    # x100 detektálás: y_pred ~ 100*y_true VAGY ~ 0.01*y_true
    x100_flag = 0
    if y_true != 0:
        if approx_ratio(y_pred, y_true, 100.0, X100_TOL) or approx_ratio(y_pred, y_true, 0.01, X100_TOL):
            x100_flag = 1

    return {
        "percent_flag": percent_flag,
        "ratio_flag": ratio_flag,
        "diff_flag": diff_flag,
        "div_flag": div_flag,
        "rounding_flag": rounding_flag,
        "has_thousand_sep": has_thousand_sep,
        "decimal_comma": decimal_comma,
        "has_percent_sign": has_percent_sign,
        "magnitude_bucket": mag_bucket,
        "rel_err_bucket": rel_err_bucket,
        "sign_err": sign_err,
        "x100_flag": x100_flag,
        "rel_err": rel_err,                 # gyakran hasznos megtartani
    }


In [15]:
"""
Question clustering (text-only, English questions)
--------------------------------------------------
Two embedding backends:
  1) TF-IDF (word + character n-grams)  — no extra deps, strong baseline
  2) Sentence-BERT (if installed)       — better semantics; auto fallback if missing

Clustering methods:
  - Agglomerative with cosine distance + auto threshold search (silhouette)
  - Optional HDBSCAN if installed (set method="hdbscan")

Usage (minimal):
  python question_clustering_text_only.py --in questions.txt --out clusters.csv

Each line in questions.txt is one question.

You can also pass questions via Python API (see bottom for example).
"""

from __future__ import annotations
import argparse
import csv
import json
import math
import os
import re
import sys
from dataclasses import dataclass
from typing import Iterable, List, Optional, Tuple

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_distances
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA

# Optional deps
try:
    import hdbscan  # type: ignore
except Exception:
    hdbscan = None

try:
    from sentence_transformers import SentenceTransformer
except Exception:
    SentenceTransformer = None  # type: ignore


# ---------------------------
# Text cleaning / preproc
# ---------------------------
_punct_norm = str.maketrans({
    "\u2013": "-",  # en dash
    "\u2014": "-",  # em dash
    "\u2019": "'",  # right single quote
    "\u2018": "'",
    "\u201c": '"',
    "\u201d": '"',
})

NUM_TOKEN = "<NUM>"
PCT_TOKEN = "<PCT>"

_num_re = re.compile(r"(?<!\w)(?:\d+[\d,]*\.?\d*)(?!\w)")
_pct_re = re.compile(r"(\d+[\d,]*\.?\d*)\s*%")


def clean_question(q: str) -> str:
    """Lightweight normalization suitable for English numeric QA questions."""
    s = q.strip().translate(_punct_norm)
    s = s.lower()
    # Normalize percentages before numbers so % doesn't double-replace
    s = _pct_re.sub(PCT_TOKEN, s)
    s = _num_re.sub(NUM_TOKEN, s)
    # Collapse extra whitespace
    s = re.sub(r"\s+", " ", s)
    return s


# ---------------------------
# Embedding backends
# ---------------------------
@dataclass
class EmbeddingConfig:
    backend: str = "tfidf"  # or "sbert"
    sbert_model: str = "all-MiniLM-L6-v2"


def embed_texts(texts: List[str], cfg: EmbeddingConfig) -> np.ndarray:
    if cfg.backend == "sbert":
        if SentenceTransformer is None:
            print("[warn] sentence-transformers not installed; falling back to TF-IDF.")
        else:
            model = SentenceTransformer(cfg.sbert_model)
            emb = model.encode(texts, show_progress_bar=False, normalize_embeddings=True)
            return np.asarray(emb)
    # TF-IDF baseline (word + char n-grams)
    vect = TfidfVectorizer(
        preprocessor=None,
        tokenizer=None,
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_df=0.95,
        lowercase=False,
    )
    X_char = vect.fit_transform(texts)
    # Word-level TF-IDF to complement char-level
    vect_w = TfidfVectorizer(
        ngram_range=(1, 2),
        stop_words="english",
        min_df=2,
        max_df=0.95,
        lowercase=False,
    )
    X_word = vect_w.fit_transform(texts)
    # Concatenate sparse matrices
    from scipy.sparse import hstack

    X = hstack([X_char, X_word]).tocsr()
    # Convert to dense only for algorithms that need it; Agglo works with distances
    # We'll return a dense array of L2-normalized rows for safety.
    # NOTE: For large datasets, keep as sparse and compute distances on-the-fly.
    X_dense = X.astype(np.float32)
    # Row-normalize
    norms = np.sqrt(X_dense.multiply(X_dense).sum(axis=1)).A1 + 1e-12
    X_dense = X_dense.multiply(1.0 / norms[:, None]).toarray()
    return X_dense


# ---------------------------
# Clustering
# ---------------------------
@dataclass
class ClusterConfig:
    method: str = "agglomerative"  # "agglomerative" | "hdbscan"
    min_cluster_size: int = 8  # used by HDBSCAN or to bound threshold search
    max_k_guess: int = 30  # for threshold search upper bound heuristic


def cluster_agglomerative(X: np.ndarray, min_cluster_size: int = 8) -> Tuple[np.ndarray, float]:
    """Cosine-distance Agglomerative with automatic distance threshold via silhouette.
    Returns (labels, best_threshold).
    """
    # If X is sparse, pairwise functions can handle it; distances will be dense
    D = cosine_distances(X)
    np.fill_diagonal(D, 0.0)

    # Search thresholds that yield between 2 and ~N/min_cluster_size clusters
    n = X.shape[0]
    max_clusters = max(2, n // max(2, min_cluster_size))
    # Candidate thresholds: quantiles of pairwise distances
    tri = D[np.triu_indices(n, k=1)]
    qs = np.linspace(0.3, 0.9, 13)  # conservative range for cosine distances
    cands = np.quantile(tri, qs)

    best_s = -1.0
    best_labels = np.full(n, -1, dtype=int)
    best_thr = float(cands[0])

    for thr in cands:
        # Fit with distance_threshold → n_clusters=None triggers tree cut by threshold
        model = AgglomerativeClustering(
            metric="precomputed",
            linkage="average",
            distance_threshold=thr,
            n_clusters=None,
        )
        labels = model.fit_predict(D)
        k = len(set(labels)) - (1 if -1 in labels else 0)
        if k < 2 or k > max_clusters:
            continue
        try:
            # Use precomputed distance matrix for silhouette
            s = silhouette_score(D, labels, metric="precomputed")
        except Exception:
            continue
        if s > best_s:
            best_s, best_labels, best_thr = s, labels, float(thr)

    # Fallback: force 2 clusters if everything failed
    if best_s < 0:
        model = AgglomerativeClustering(metric="precomputed", linkage="average", n_clusters=2)
        best_labels = model.fit_predict(D)
        best_thr = float(np.median(tri))

    return best_labels, best_thr


def cluster_hdbscan(X: np.ndarray, min_cluster_size: int = 8) -> Tuple[np.ndarray, Optional[float]]:
    if hdbscan is None:
        print("[warn] hdbscan not installed; falling back to agglomerative.")
        return cluster_agglomerative(X, min_cluster_size)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=None, metric="euclidean")
    labels = clusterer.fit_predict(X)
    return labels, None


# ---------------------------
# Public API
# ---------------------------
@dataclass
class Result:
    labels: np.ndarray
    embedding_backend: str
    cluster_method: str
    extra: dict


def cluster_questions(
    questions: List[str],
    embedding_backend: str = "tfidf",
    cluster_method: str = "agglomerative",
    min_cluster_size: int = 8,
) -> Result:
    if not questions:
        raise ValueError("Empty question list.")
    cleaned = [clean_question(q) for q in questions]
    X = embed_texts(cleaned, EmbeddingConfig(backend=embedding_backend))
    if cluster_method == "hdbscan":
        labels, thr = cluster_hdbscan(X, min_cluster_size=min_cluster_size)
    else:
        labels, thr = cluster_agglomerative(X, min_cluster_size=min_cluster_size)

    return Result(
        labels=np.asarray(labels),
        embedding_backend=embedding_backend,
        cluster_method=cluster_method,
        extra={"threshold": thr},
    )


# ---------------------------
# I/O helpers
# ---------------------------

def read_lines(path: str) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        return [ln.rstrip("\n") for ln in f if ln.strip()]


def write_clusters_csv(path: str, questions: List[str], labels: np.ndarray) -> None:
    rows = [(int(lbl), q) for q, lbl in zip(questions, labels)]
    rows.sort(key=lambda x: (x[0], x[1]))
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["cluster", "question"])
        for lbl, q in rows:
            w.writerow([lbl, q])


def preview_clusters(questions: List[str], labels: np.ndarray, max_examples: int = 5) -> str:
    from collections import defaultdict
    bucket = defaultdict(list)
    for q, l in zip(questions, labels):
        bucket[int(l)].append(q)
    parts = []
    for k in sorted(bucket.keys()):
        ex = "\n".join("  - " + s for s in bucket[k][:max_examples])
        parts.append(f"Cluster {k} (n={len(bucket[k])}):\n{ex}")
    return "\n\n".join(parts)


# ---------------------------
# Example (Python API)
# ---------------------------
# questions = [
#     "What is the percentage increase from 80 to 92?",
#     "What percent of 250 is 40?",
#     "By how much did sales grow year-over-year?",
#     "What is 17 + 24?",
#     "Compute the ratio of boys to girls if there are 12 boys and 8 girls.",
#     "How many percentage points is 12% vs 9%?",
# ]
# res = cluster_questions(questions, embedding_backend="tfidf", cluster_method="agglomerative")
# print(preview_clusters(questions, res.labels))


# --- New: feature flags + vectorization pipeline (pandas + DictVectorizer) ---
# This extends the script with a full example of building a feature table from
# question text + y_true/y_pred + calc_pattern, then clustering.

import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from scipy.sparse import hstack

# Reuse: extract_flags, clean_question, cluster_agglomerative / cluster_hdbscan from above


def build_feature_matrix(df: pd.DataFrame,
                         use_tfidf_text: bool = True,
                         use_calc_pattern_onehot: bool = True,
                         text_col: str = "question",
                         true_col: str = "answer",
                         pred_col: str = "pred",
                         pattern_col: str = "calc_pattern"):
    """
    Input df columns:
        - question (str): the raw English question
        - y_true (float)
        - y_pred (float)
        - calc_pattern (str) optional, e.g. "#-#", "(#-#)/#"
    Returns:
        X (scipy.sparse/dense): combined feature matrix
        parts (dict): named parts for debugging
    """
    # --- 1) flag dicts from question + numbers ---
    flag_dicts = [extract_flags(q, float(y), float(yhat), str(df.get(pattern_col, [""])[i])
                                if pattern_col in df.columns else "")
                  for i, (q, y, yhat) in enumerate(zip(df[text_col], df[true_col], df[pred_col]))]

    dv = DictVectorizer(sparse=True)
    X_flags = dv.fit_transform(flag_dicts)

    # --- 2) text TF-IDF (char + word) ---
    X_text = None
    if use_tfidf_text:
        tf_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2, max_df=0.95)
        tf_word = TfidfVectorizer(ngram_range=(1,2), stop_words="english", min_df=2, max_df=0.95)
        Q_clean = [clean_question(q) for q in df[text_col].fillna("")]
        Xc = tf_char.fit_transform(Q_clean)
        Xw = tf_word.fit_transform(Q_clean)
        X_text = hstack([Xc, Xw])

    # --- 3) calc_pattern one-hot ---
    X_pat = None
    enc = None
    if use_calc_pattern_onehot and pattern_col in df.columns:
        enc = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
        X_pat = enc.fit_transform(df[[pattern_col]].fillna(""))

    # --- 4) combine parts ---
    parts = [X_flags]
    if X_text is not None:
        parts.append(X_text)
    if X_pat is not None:
        parts.append(X_pat)

    X = parts[0]
    for P in parts[1:]:
        X = hstack([X, P]).tocsr()

    return X, {
        "dict_vectorizer": dv,
        "text_char_word": (tf_char if use_tfidf_text else None, tf_word if use_tfidf_text else None),
        "pattern_encoder": enc,
        "flag_feature_names": dv.get_feature_names_out(),
    }


def cluster_dataframe(df: pd.DataFrame,
                      method: str = "agglomerative",
                      min_cluster_size: int = 8):
    """High-level convenience: build features and cluster; returns labels and parts."""
    X, parts = build_feature_matrix(df)
    if method == "hdbscan":
        labels, thr = cluster_hdbscan(X, min_cluster_size=min_cluster_size)
    else:
        labels, thr = cluster_agglomerative(X, min_cluster_size=min_cluster_size)
    return labels, parts, thr






In [21]:
import cga_utils

errors = pd.read_csv('res/e38_18r.csv').query('exact_match == False')

labels, parts, thr = cluster_dataframe(errors, method="agglomerative", min_cluster_size=2)
df_out = df.copy()
df_out["cluster"] = labels

print("Chosen threshold:", thr)
print(df_out.sort_values("cluster"))

KeyError: 0